In [15]:
import pandas as pd
import numpy as np
import yaml

## Get global mean and std for stdscaling

In [16]:
df_train = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/cnn-baseline/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_train_NEW.csv").sort_values(by=["shot_idx", "variable"])

In [17]:
df_train[df_train["shot_id"]==19454]

,shot_idx,shot_id,variable,n_dim_shot,mean,variance
230170,5901,19454,equilibrium-beta_normal,106.0,-9.177884e+00,8.038097e+01
230169,5901,19454,equilibrium-beta_pol,106.0,-1.818459e+00,2.065338e+00
230168,5901,19454,equilibrium-beta_tor,106.0,-1.631531e+01,3.050829e+02
230172,5901,19454,equilibrium-bphi_rmag,106.0,-1.056248e+00,2.124651e-01
230171,5901,19454,equilibrium-bvac_rmag,106.0,-5.382474e-01,4.125114e-03
230156,5901,19454,equilibrium-elongation,106.0,1.432538e+00,8.920645e-03
230157,5901,19454,equilibrium-elongation_axis,106.0,1.279411e+00,7.187563e-02
230160,5901,19454,equilibrium-lcfs_r,18020.0,6.781615e-01,1.146467e-01
230161,5901,19454,equilibrium-lcfs_z,18020.0,2.613037e-05,2.521327e-01
230165,5901,19454,equilibrium-magnetic_axis_r,106.0,7.038587e-01,6.793435e-03


In [18]:
# GLOBAL MEAN BASED ON TRAIN
global_mean = (
    df_train
    .groupby("variable")[["n_dim_shot", "mean", ]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean")
)
df_train_with_group_mean = df_train.join(global_mean, on="variable")

# GLOBAL VARIANCE BASED ON TRAIN
global_variance = (
    df_train_with_group_mean
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance")
)
df_train_with_group_mean_and_variance = df_train_with_group_mean.join(global_variance, on="variable")
df_train_with_group_mean_and_variance.head()

,shot_idx,shot_id,variable,n_dim_shot,mean,variance,global_mean,global_variance
31,0,21719,equilibrium-beta_normal,109.0,0.509525,0.157241,1.011904,2.248063
30,0,21719,equilibrium-beta_pol,109.0,0.118568,0.018020,0.212163,0.119754
29,0,21719,equilibrium-beta_tor,109.0,1.070841,0.414899,2.413701,14.475489
33,0,21719,equilibrium-bphi_rmag,109.0,-0.554288,0.001070,-0.513486,0.039480
32,0,21719,equilibrium-bvac_rmag,109.0,-0.474535,0.001585,-0.444512,0.028251


#### Remove z-6 outliers from computation

In [19]:
# Z-SCORE COMPUTATION TRAIN
# df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) / np.sqrt(df_train_with_group_mean_and_variance["n_dim_shot"]) )
df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) )

df_train_with_group_mean_and_variance

# COUNTING OUTLIERS
print( f'Count of z-6 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 6 )} out of {len(df_train_with_group_mean_and_variance)}')
print( f'Count of z-12 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 12 )} out of {len(df_train_with_group_mean_and_variance)}')

df_train_with_group_mean_and_variance["outlier_z_6"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 6 
df_train_with_group_mean_and_variance["outlier_z_12"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 12

df_train_extended = df_train_with_group_mean_and_variance.copy()
df_train_with_group_mean_and_variance.sort_values('z_score').dropna()

Count of z-6 outliers 84 out of 361101
Count of z-12 outliers 11 out of 361101


,shot_idx,shot_id,variable,n_dim_shot,mean,variance,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
29240,749,19382,equilibrium-beta_tor,90.0,-119.605869,3340.857403,2.413701,14.475489,-32.071027,True,True
23007,589,13298,soft_x_rays-horizontal_cam_upper,437760.0,-0.738288,1.708373,0.003237,0.000732,-27.406186,True,True
284885,7304,19374,equilibrium-beta_tor,90.0,-95.180347,823.812779,2.413701,14.475489,-25.651142,True,True
31424,805,19391,equilibrium-beta_tor,78.0,-50.513519,6.095461,2.413701,14.475489,-13.911131,True,True
223382,5727,19413,equilibrium-beta_tor,78.0,-46.679489,377.113179,2.413701,14.475489,-12.903414,True,True
...,...,...,...,...,...,...,...,...,...,...,...
247484,6345,12231,equilibrium-beta_tor,54.0,61.114025,958.049300,2.413701,14.475489,15.428506,True,True
20855,534,12228,equilibrium-beta_tor,54.0,62.909467,1264.040519,2.413701,14.475489,15.900411,True,True
55052,1411,12184,equilibrium-x_point_r,156.0,18.009783,721.322216,0.508192,0.829638,19.214688,True,True
252977,6486,12168,equilibrium-x_point_r,136.0,18.505081,863.610863,0.508192,0.829638,19.758466,True,True


In [20]:
# import matplotlib.pyplot as plt

# plt.hist(df_train_with_group_mean_and_variance["z_score"], bins=30)
# plt.xlabel("z_score")
# plt.ylabel("Frequency")
# plt.title("Histogram of z-scores")
# plt.show()

In [21]:
df_train_extended.head()

,shot_idx,shot_id,variable,n_dim_shot,mean,variance,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
31,0,21719,equilibrium-beta_normal,109.0,0.509525,0.157241,1.011904,2.248063,-0.335063,False,False
30,0,21719,equilibrium-beta_pol,109.0,0.118568,0.018020,0.212163,0.119754,-0.270463,False,False
29,0,21719,equilibrium-beta_tor,109.0,1.070841,0.414899,2.413701,14.475489,-0.352951,False,False
33,0,21719,equilibrium-bphi_rmag,109.0,-0.554288,0.001070,-0.513486,0.039480,-0.205348,False,False
32,0,21719,equilibrium-bvac_rmag,109.0,-0.474535,0.001585,-0.444512,0.028251,-0.178628,False,False


In [22]:
# OG global mean and variance
global_mean = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "mean"]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean")
)
global_variance = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance")
)

# MEAN AND VARIANCE COMPUTATION WITHOUT OUTLIERS Z-6
# global mean and variance without the 6-z outliers
global_mean_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "mean"]]
    .apply(lambda g: (g["mean"] * g["n_dim_shot"]).sum() / g["n_dim_shot"].sum())
    .rename("global_mean_no_z_6")
)
global_variance_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( g["n_dim_shot"] * g["variance"] + g["n_dim_shot"] * (g["mean"]-g["global_mean"])**2 ).sum() / g["n_dim_shot"].sum())
    .rename("global_variance_no_z_6")
)


In [23]:
# Merge all the stats into a single DataFrame
df_stats = pd.DataFrame({
    "variable": global_mean.index,
    "mean_all": global_mean.values,
    "std_all": np.sqrt(global_variance.values),
    "mean_no_outliers_z6": global_mean_no_z_6.values,
    "std_no_outliers_z6": np.sqrt(global_variance_no_z_6.values),
})

# Build the dictionary for YAML
final_dict = {}
for _, row in df_stats.iterrows():
    var = row["variable"]
    final_dict[var] = {
        "mean": {
            "all": row["mean_all"],
            "no_outliers_z6": row["mean_no_outliers_z6"],
            # "no_outliers_z12": row["mean_no_outliers_z12"]
        },
        "std": {
            "all": row["std_all"],
            "no_outliers_z6": row["std_no_outliers_z6"],
            # "no_outliers_z12": row["std_no_outliers_z12"]
        }
    }

# Write to YAML
with open("mean_std_train.yaml", "w") as f:
    yaml.dump(final_dict, f, sort_keys=False)


## Remove outliers from train, val, test

In [24]:
df_train = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/cnn-baseline/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_train_NEW.csv").sort_values(by=["shot_idx", "variable"])
df_val = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/cnn-baseline/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_val_NEW.csv").sort_values(by=["shot_idx", "variable"])
df_test = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/cnn-baseline/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_test_NEW.csv").sort_values(by=["shot_idx", "variable"])

df_all = pd.concat([df_train, df_val, df_test])

In [25]:
df_all_with_stats = df_all.join(global_mean_no_z_6, on="variable").join(global_variance_no_z_6, on="variable")
df_all_with_stats

df_all_with_stats["z_score"] = (df_all_with_stats["mean"] - df_all_with_stats["global_mean_no_z_6"]) / ( np.sqrt(df_all_with_stats["global_variance_no_z_6"]) )

df_all_with_stats["outlier_z_12"] = abs(df_all_with_stats["z_score"]) > 12

print( f'Count of z-12 outliers {sum ( df_all_with_stats["outlier_z_12"] ) } out of {len(df_all_with_stats)}')
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() )} variables {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() } (out of {len( df_all_with_stats["variable"].unique()) })' )
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique() )} shots {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique()} (out of {len( df_all_with_stats["shot_id"].unique()) })' )


Count of z-12 outliers 26 out of 451347
Spanning 4 variables ['equilibrium-beta_tor' 'soft_x_rays-horizontal_cam_upper'
 'equilibrium-x_point_r' 'equilibrium-beta_normal'] (out of 39)
Spanning 26 shots [12228 13298 19382 19391 12184 15926 19415 19375 11892 12185 12053 19413
 19409 12231 11901 12168 19393 19374 19408 19401 19388 19386 19410 11899
 19455 19387] (out of 11573)


In [26]:
# import matplotlib.pyplot as plt

# plt.hist(
#     df_all_with_stats[df_all_with_stats["outlier_z_12"]]["shot_id"],
#     bins=100,
#     color="salmon",
#     edgecolor="black"
# )
# plt.xlabel("shot_id")
# plt.ylabel("Number of outliers")
# plt.show()


In [27]:
outlier_dict = (
    df_all_with_stats[df_all_with_stats["outlier_z_12"]].groupby("shot_id")["variable"]
    .apply(list)
    .to_dict()
)
print(outlier_dict)

with open("dict_outlier_metadata.yaml", "w") as f_:
    yaml.dump(outlier_dict, f_, sort_keys=False)

{11892: ['equilibrium-x_point_r'], 11899: ['equilibrium-x_point_r'], 11901: ['equilibrium-x_point_r'], 12053: ['equilibrium-x_point_r'], 12168: ['equilibrium-x_point_r'], 12184: ['equilibrium-x_point_r'], 12185: ['equilibrium-x_point_r'], 12228: ['equilibrium-beta_tor'], 12231: ['equilibrium-beta_tor'], 13298: ['soft_x_rays-horizontal_cam_upper'], 15926: ['equilibrium-beta_normal'], 19374: ['equilibrium-beta_tor'], 19375: ['equilibrium-beta_tor'], 19382: ['equilibrium-beta_tor'], 19386: ['equilibrium-beta_tor'], 19387: ['equilibrium-beta_normal'], 19388: ['equilibrium-beta_tor'], 19391: ['equilibrium-beta_tor'], 19393: ['equilibrium-beta_tor'], 19401: ['equilibrium-beta_tor'], 19408: ['equilibrium-beta_normal'], 19409: ['equilibrium-beta_tor'], 19410: ['equilibrium-beta_tor'], 19413: ['equilibrium-beta_tor'], 19415: ['equilibrium-beta_normal'], 19455: ['equilibrium-beta_tor']}


In [28]:
outlier_dict.keys()

dict_keys([11892, 11899, 11901, 12053, 12168, 12184, 12185, 12228, 12231, 13298, 15926, 19374, 19375, 19382, 19386, 19387, 19388, 19391, 19393, 19401, 19408, 19409, 19410, 19413, 19415, 19455])